# ISIC 2016 VGG-U-Net Segmentation and Explainability

**Dataset:** ISIC 2016  
**Task:** Skin lesion segmentation  
**Model:** VGG-U-Net  
**Methods:** Dice and IoU evaluation, Grad-CAM, Grad-CAM++, boundary-focused analysis and SHAP

This notebook evaluates lesion segmentation and investigates how different explanation targets highlight the lesion and its boundaries. It was developed in Google Colab; dataset, checkpoint and output paths must be adjusted before execution.

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
  import os, glob, random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg16_bn, VGG16_BN_Weights


In [ ]:

!pip -q install shap

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn


model.eval()


class ScalarOutputModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        logits = self.base_model(x)
        return logits.sum(dim=(1, 2, 3)).unsqueeze(1)


scalar_model = ScalarOutputModel(model).to(device)


torch.set_grad_enabled(True)


background_sample, _ = next(iter(train_loader))
background_sample = background_sample[0:1].to(device)
_sample)

print("SHAP DeepExplainer created with scalar output wrapper.")

In [ ]:
import os

root = "/content/isic2016"
if not os.path.exists(root):
    os.makedirs(root, exist_ok=True)

if not os.path.exists(f"{root}/train_images") or not os.path.exists(f"{root}/train_masks"):
    %cd /content/isic2016
    !wget -q https://isic-archive.s3.amazonaws.com/challenges/2016/ISBI2016_ISIC_Part1_Training_Data.zip -O train_images.zip
    !wget -q https://isic-archive.s3.amazonaws.com/challenges/2016/ISBI2016_ISIC_Part1_Training_GroundTruth.zip -O train_masks.zip
    !unzip -q train_images.zip -d train_images
    !unzip -q train_masks.zip -d train_masks
    print("Downloaded + unzipped.")
else:
    print("Dataset folders already exist.")


In [ ]:
img_paths  = sorted(glob.glob("/content/isic2016/train_images/**/*.jpg", recursive=True))
mask_paths = sorted(glob.glob("/content/isic2016/train_masks/**/*.png", recursive=True))

def stem(p):
    return os.path.splitext(os.path.basename(p))[0]

mask_map = {stem(p).replace("_segmentation","").replace("_Segmentation",""): p for p in mask_paths}

pairs = []
for ip in img_paths:
    sid = stem(ip)
    mp = mask_map.get(sid, None)
    if mp is not None:
        pairs.append((ip, mp))

print("images:", len(img_paths), "| masks:", len(mask_paths), "| paired:", len(pairs))
print("example:", pairs[0])


In [ ]:
IMG_SIZE = 256
BATCH = 4

img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
])

mask_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=Image.NEAREST),
])

class ISICSeg(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ip, mp = self.pairs[idx]
        img = Image.open(ip).convert("RGB")
        msk = Image.open(mp).convert("L")

        x = img_tf(img)

        msk = mask_tf(msk)
        m = (np.array(msk) > 0).astype(np.float32)
        y = torch.from_numpy(m)[None, ...]  # (1,H,W)

        return x, y

random.shuffle(pairs)
n = len(pairs)
train_pairs = pairs[:int(0.8*n)]
val_pairs   = pairs[int(0.8*n):]

train_ds = ISICSeg(train_pairs)
val_ds   = ISICSeg(val_pairs)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_loader))
print("xb:", xb.shape, "yb:", yb.shape)


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=False),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=False),
        )
    def forward(self, x): return self.net(x)

class VGGUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = VGG16_BN_Weights.IMAGENET1K_V1 if pretrained else None
        vgg = vgg16_bn(weights=weights).features


        self.enc1 = vgg[:6]
        self.pool1 = vgg[6]
        self.enc2 = vgg[7:13]
        self.pool2 = vgg[13]
        self.enc3 = vgg[14:23]
        self.pool3 = vgg[23]
        self.enc4 = vgg[24:33]
        self.pool4 = vgg[33]
        self.enc5 = vgg[34:43]
        self.pool5 = vgg[43]


        self.mid = DoubleConv(512, 512)


        self.up5 = nn.ConvTranspose2d(512, 512, 2, stride=2)
        self.dec5 = DoubleConv(512+512, 512)

        self.up4 = nn.ConvTranspose2d(512, 512, 2, stride=2)
        self.dec4 = DoubleConv(512+512, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(256+256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(128+128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(64+64, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        x1 = self.enc1(x)              # 64
        x2 = self.enc2(self.pool1(x1)) # 128
        x3 = self.enc3(self.pool2(x2)) # 256
        x4 = self.enc4(self.pool3(x3)) # 512
        x5 = self.enc5(self.pool4(x4)) # 512
        x6 = self.pool5(x5)            # 512

        m  = self.mid(x6)

        d5 = self.up5(m)
        d5 = self.dec5(torch.cat([d5, x5], dim=1))

        d4 = self.up4(d5)
        d4 = self.dec4(torch.cat([d4, x4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, x3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, x2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, x1], dim=1))

        return self.out(d1)


def replace_relu_inplace(module):
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU):
            setattr(module, name, nn.ReLU(inplace=False))
        else:
            replace_relu_inplace(child)

model = VGGUNet(pretrained=True).to(device)
replace_relu_inplace(model)
print("model ready and ReLUs set to inplace=False")

In [ ]:
bce = nn.BCEWithLogitsLoss()

def soft_dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2*(probs*targets).sum(dim=(1,2,3)) + eps
    den = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) + eps
    return 1 - (num/den).mean()

def loss_fn(logits, targets):
    return bce(logits, targets) + soft_dice_loss(logits, targets)

def dice_iou(logits, targets, eps=1e-7):
    probs = (torch.sigmoid(logits) > 0.5).float()
    inter = (probs * targets).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (union + eps)

    union_iou = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) - inter
    iou = (inter + eps) / (union_iou + eps)
    return dice.mean().item(), iou.mean().item()


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_dice, total_iou, n = 0.0, 0.0, 0.0, 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        if train:
            opt.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train), torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(xb)
            loss = loss_fn(logits, yb)

        if train:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        d, j = dice_iou(logits.detach(), yb)
        bs = xb.size(0)

        total_loss += float(loss.item()) * bs
        total_dice += d * bs
        total_iou  += j * bs
        n += bs

    return {
        "loss": total_loss / n,
        "dice": total_dice / n,
        "iou": total_iou / n
    }


best_val = -1
SAVE_DIR = "/content/vgg_unet_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

best_path = os.path.join(SAVE_DIR, "vgg_unet_best.pth")

EPOCHS = 10
SAVE_EPOCHS = [1, 4, 8]

history = []

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)

    history.append({
        "epoch": epoch,
        "train_loss": tr["loss"],
        "train_dice": tr["dice"],
        "train_iou": tr["iou"],
        "val_loss": va["loss"],
        "val_dice": va["dice"],
        "val_iou": va["iou"]
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={tr['loss']:.4f} | train_dice={tr['dice']:.4f} | train_iou={tr['iou']:.4f} | "
        f"val_loss={va['loss']:.4f} | val_dice={va['dice']:.4f} | val_iou={va['iou']:.4f}"
    )

    if epoch in SAVE_EPOCHS:
        epoch_path = os.path.join(SAVE_DIR, f"vgg_unet_epoch_{epoch}.pth")
        torch.save(model.state_dict(), epoch_path)
        print("Saved checkpoint:", epoch_path)

    if va["dice"] > best_val:
        best_val = va["dice"]
        torch.save(model.state_dict(), best_path)
        print("Saved best model:", best_path, "| best val dice:", best_val)

print("Training finished.")
print("Best val dice:", best_val)

In [ ]:

denorm = transforms.Normalize(
    mean=(-0.485/0.229, -0.456/0.224, -0.406/0.225),
    std=(1/0.229, 1/0.224, 1/0.225)
)

model.eval()
xb, yb = next(iter(val_loader))
xb = xb.to(device)
with torch.no_grad():
    logits = model(xb)
    prob = torch.sigmoid(logits).cpu().numpy()

plt.figure(figsize=(12,8))
for i in range(min(3, xb.size(0))):
    img = denorm(xb[i].cpu()).permute(1,2,0).clamp(0,1).numpy()
    gt  = yb[i,0].numpy()
    pr  = prob[i,0]

    plt.subplot(3,3,3*i+1); plt.imshow(img); plt.axis("off"); plt.title("Input")
    plt.subplot(3,3,3*i+2); plt.imshow(gt, cmap="gray"); plt.axis("off"); plt.title("GT mask")
    plt.subplot(3,3,3*i+3); plt.imshow(img); plt.imshow(pr, cmap="Reds", alpha=0.35); plt.axis("off"); plt.title("Pred overlay")
plt.tight_layout()
plt.show()


In [ ]:
!pip -q install grad-cam opencv-python


In [ ]:
import torch.nn as nn

def get_last_conv(module):
    last = None
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last

target_layer = get_last_conv(model.enc5)
print("Target conv layer:", target_layer)


In [ ]:
import numpy as np
import torch
import cv2

class SegmentationMaskTargetMean:
    def __init__(self, mask_2d):
        self.mask = torch.from_numpy(mask_2d.astype(np.float32))
        self.den = float(mask_2d.sum() + 1e-6)

    def __call__(self, model_output):
        # model_output: (1,1,H,W) logits
        logits = model_output[0, 0]
        mask = self.mask.to(logits.device)
        return (logits * mask).sum() / self.den


In [ ]:
import torch
import torch.nn as nn

print("grad enabled:", torch.is_grad_enabled())


trainable = sum(p.requires_grad for p in model.parameters())
total = sum(1 for _ in model.parameters())
print("trainable params:", trainable, "/", total)


def get_last_conv(module):
    last = None
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last

target_layer = get_last_conv(model.enc5)
print("target_layer:", target_layer)
if target_layer is not None:
    print("target_layer.requires_grad(weight):", target_layer.weight.requires_grad)


In [ ]:
target_layer = model.dec1.net[3]
print("new target_layer:", target_layer)


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


def replace_relu_inplace(module):
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU):
            setattr(module, name, nn.ReLU(inplace=False))
        else:
            replace_relu_inplace(child)

def load_model_for_gradcam():
    temp_model = VGGUNet(pretrained=False).to(device)
    replace_relu_inplace(temp_model)

    temp_model.load_state_dict(torch.load(best_path, map_location=device), strict=False)
    temp_model.eval()
    return temp_model


fresh_model = load_model_for_gradcam()


xb, yb = next(iter(val_loader))
xb = xb.to(device)
yb = yb.to(device)

i = 0
x1 = xb[i:i+1]
gt = yb[i,0].detach().cpu().numpy()


with torch.no_grad():
    logits = fresh_model(x1)
    prob = torch.sigmoid(logits)[0,0].detach().cpu().numpy()
    pred_mask = (prob > 0.5).astype(np.float32)


import cv2, numpy as np
gt_u8 = (gt*255).astype(np.uint8)
edges = cv2.Canny(gt_u8, 50, 150)
band_gt = cv2.dilate(edges, np.ones((9,9), np.uint8), iterations=1)
band_gt = (band_gt > 0).astype(np.float32)


class SegmentationMaskTargetMean:
    def __init__(self, mask_2d):
        self.mask = torch.from_numpy(mask_2d.astype(np.float32))
        self.den = float(mask_2d.sum() + 1e-6)
    def __call__(self, model_output):
        logits = model_output[0,0]
        mask = self.mask.to(logits.device)
        return (logits*mask).sum() / self.den


target_layer = fresh_model.dec2.net[3]

cam = GradCAM(model=fresh_model, target_layers=[target_layer])

cam_gt   = cam(input_tensor=x1, targets=[SegmentationMaskTargetMean(gt)])[0]
cam_pred = cam(input_tensor=x1, targets=[SegmentationMaskTargetMean(pred_mask)])[0]
cam_edge = cam(input_tensor=x1, targets=[SegmentationMaskTargetMean(band_gt)])[0]

print("cam_gt   min/max/mean:", float(cam_gt.min()), float(cam_gt.max()), float(cam_gt.mean()))
print("cam_pred min/max/mean:", float(cam_pred.min()), float(cam_pred.max()), float(cam_pred.mean()))
print("cam_edge min/max/mean:", float(cam_edge.min()), float(cam_edge.max()), float(cam_edge.mean()))

In [ ]:
import matplotlib.pyplot as plt
img = denorm(x1[0].detach().cpu()).permute(1,2,0).clamp(0,1).numpy()

overlay_gt   = show_cam_on_image(img.astype(np.float32), cam_gt,   use_rgb=True)
overlay_pred = show_cam_on_image(img.astype(np.float32), cam_pred, use_rgb=True)
overlay_edge = show_cam_on_image(img.astype(np.float32), cam_edge, use_rgb=True)

plt.figure(figsize=(15,9))
plt.subplot(2,3,1); plt.imshow(img); plt.axis("off"); plt.title("Input")
plt.subplot(2,3,2); plt.imshow(gt, cmap="gray"); plt.axis("off"); plt.title("GT mask")
plt.subplot(2,3,3); plt.imshow(prob, cmap="gray"); plt.axis("off"); plt.title("Pred prob")
plt.subplot(2,3,4); plt.imshow(overlay_gt); plt.axis("off"); plt.title("GradCAM (GT mean)")
plt.subplot(2,3,5); plt.imshow(overlay_pred); plt.axis("off"); plt.title("GradCAM (Pred mean)")
plt.subplot(2,3,6); plt.imshow(overlay_edge); plt.axis("off"); plt.title("GradCAM (Boundary band)")
plt.tight_layout(); plt.show()


In [ ]:
band_gt = cv2.dilate(edges, np.ones((5,5), np.uint8), iterations=1)
band_gt = (band_gt > 0).astype(np.float32)
print("band_gt coverage:", band_gt.mean(), "pixels:", int(band_gt.sum()))


In [ ]:
out_path = "/content/vgg_unet_gradcam_triplet.png"
plt.figure(figsize=(15,9))

plt.subplot(2,3,1); plt.imshow(img); plt.axis("off"); plt.title("Input")
plt.subplot(2,3,2); plt.imshow(gt, cmap="gray"); plt.axis("off"); plt.title("GT mask")
plt.subplot(2,3,3); plt.imshow(prob, cmap="gray"); plt.axis("off"); plt.title("Pred prob")

plt.subplot(2,3,4); plt.imshow(overlay_gt); plt.axis("off"); plt.title("Grad-CAM++ (GT mean target)")
plt.subplot(2,3,5); plt.imshow(overlay_pred); plt.axis("off"); plt.title("Grad-CAM++ (Pred mean target)")
plt.subplot(2,3,6); plt.imshow(overlay_edge); plt.axis("off"); plt.title("Grad-CAM++ (Boundary-band GT target)")

plt.tight_layout()
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", out_path)


In [ ]:
import torch
import numpy as np

def dice_per_sample(pred_mask, gt_mask, eps=1e-7):

    inter = (pred_mask * gt_mask).sum(dim=(1,2,3))
    den = pred_mask.sum(dim=(1,2,3)) + gt_mask.sum(dim=(1,2,3))
    return (2*inter + eps) / (den + eps)

model.eval()

worst = {"dice": 1.0, "x1": None, "gt": None, "prob": None, "pred": None}
MAX_BATCHES = 30

with torch.no_grad():
    for b, (xb, yb) in enumerate(val_loader):
        if b >= MAX_BATCHES:
            break
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        prob = torch.sigmoid(logits)
        pred = (prob > 0.5).float()

        d = dice_per_sample(pred, yb)  # (B,)
        min_d, idx = d.min().item(), int(d.argmin().item())

        if min_d < worst["dice"]:
            worst["dice"] = min_d
            worst["x1"] = xb[idx:idx+1].detach()
            worst["gt"] = yb[idx,0].detach().cpu().numpy()
            worst["prob"] = prob[idx,0].detach().cpu().numpy()
            worst["pred"] = pred[idx,0].detach().cpu().numpy()

print("Worst dice found:", worst["dice"])
x1 = worst["x1"]                  # (1,3,H,W)
gt = worst["gt"]                  # (H,W)
prob = worst["prob"]              # (H,W)
pred_mask = worst["pred"].astype(np.float32)  # (H,W)

In [ ]:
import cv2
import numpy as np

gt_u8 = (gt * 255).astype(np.uint8)
k = np.ones((9,9), np.uint8)

dil = cv2.dilate(gt_u8, k, iterations=1)
ero = cv2.erode(gt_u8, k, iterations=1)
ring = ((dil > 0) & (ero == 0)).astype(np.float32)

print("ring coverage:", ring.mean(), "| pixels:", int(ring.sum()))

In [ ]:
from pytorch_grad_cam import GradCAM
import torch.nn as nn


target_layer = model.dec2.net[3]


print("Target layer:", target_layer)

In [ ]:
import torch

class SegmentationMaskTargetMean:
    def __init__(self, mask_2d):
        self.mask = torch.from_numpy(mask_2d.astype(np.float32))
        self.den = float(mask_2d.sum() + 1e-6)
    def __call__(self, model_output):
        logits = model_output[0,0]          # (H,W)
        mask = self.mask.to(logits.device)
        return (logits * mask).sum() / self.den


target_layer = fresh_model.dec2.net[3]
cam = GradCAM(model=fresh_model, target_layers=[target_layer])


fresh_model.eval()
cam_gt   = cam(input_tensor=x1, targets=[SegmentationMaskTargetMean(gt)])[0]
cam_pred = cam(input_tensor=x1, targets=[SegmentationMaskTargetMean(pred_mask)])[0]
cam_edge = cam(input_tensor=x1, targets=[SegmentationMaskTargetMean(ring)])[0]

print("means:", float(cam_gt.mean()), float(cam_pred.mean()), float(cam_edge.mean()))

In [ ]:
import numpy as np

def corr(a,b):
    a = a.flatten(); b = b.flatten()
    return float(np.corrcoef(a,b)[0,1])

print("corr(gt,pred):", corr(cam_gt, cam_pred))
print("corr(gt,edge):", corr(cam_gt, cam_edge))
print("corr(pred,edge):", corr(cam_pred, cam_edge))

diff_gt_edge = np.abs(cam_gt - cam_edge)
diff_gt_pred = np.abs(cam_gt - cam_pred)
print("mean |gt-edge|:", float(diff_gt_edge.mean()))
print("mean |gt-pred|:", float(diff_gt_pred.mean()))

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam.utils.image import show_cam_on_image


img = denorm(x1[0].detach().cpu()).permute(1,2,0).clamp(0,1).numpy()

ov_gt   = show_cam_on_image(img.astype(np.float32), cam_gt,   use_rgb=True)
ov_pred = show_cam_on_image(img.astype(np.float32), cam_pred, use_rgb=True)
ov_edge = show_cam_on_image(img.astype(np.float32), cam_edge, use_rgb=True)

plt.figure(figsize=(16,10))
plt.subplot(2,4,1); plt.imshow(img); plt.axis("off"); plt.title("Input")
plt.subplot(2,4,2); plt.imshow(gt, cmap="gray"); plt.axis("off"); plt.title("GT mask")
plt.subplot(2,4,3); plt.imshow(prob, cmap="gray"); plt.axis("off"); plt.title("Pred prob")
plt.subplot(2,4,4); plt.imshow(ring, cmap="gray"); plt.axis("off"); plt.title("Boundary ring")

plt.subplot(2,4,5); plt.imshow(ov_gt); plt.axis("off"); plt.title("GradCAM (GT mean)")
plt.subplot(2,4,6); plt.imshow(ov_pred); plt.axis("off"); plt.title("GradCAM (Pred mean)")
plt.subplot(2,4,7); plt.imshow(ov_edge); plt.axis("off"); plt.title("GradCAM (Edge ring mean)")
plt.subplot(2,4,8); plt.imshow(np.abs(cam_gt-cam_edge), cmap="magma"); plt.axis("off"); plt.title("|GT - Edge|")

plt.tight_layout(); plt.show()

In [ ]:
import cv2, numpy as np

gt_u8 = (gt*255).astype(np.uint8)
k = np.ones((9,9), np.uint8)

dil = cv2.dilate(gt_u8, k, iterations=1)
ero = cv2.erode(gt_u8, k, iterations=1)

ring = ((dil > 0) & (ero == 0)).astype(np.float32)     # boundary ring
interior = (ero > 0).astype(np.float32)                # inside area (eroded)
print("ring px:", int(ring.sum()), "| interior px:", int(interior.sum()))

In [ ]:
import torch

class BoundaryContrastTarget:
    def __init__(self, ring_2d, interior_2d):
        self.ring = torch.from_numpy(ring_2d.astype(np.float32))
        self.interior = torch.from_numpy(interior_2d.astype(np.float32))
        self.den_r = float(ring_2d.sum() + 1e-6)
        self.den_i = float(interior_2d.sum() + 1e-6)

    def __call__(self, model_output):
        logits = model_output[0,0]
        r = self.ring.to(logits.device)
        i = self.interior.to(logits.device)
        return (logits*r).sum()/self.den_r - (logits*i).sum()/self.den_i

In [ ]:
cam_edge_contrast = cam(input_tensor=x1, targets=[BoundaryContrastTarget(ring, interior)])[0]

import numpy as np
def corr(a,b):
    a=a.flatten(); b=b.flatten()
    return float(np.corrcoef(a,b)[0,1])

print("corr(gt, edge_contrast):", corr(cam_gt, cam_edge_contrast))
print("mean |gt - edge_contrast|:", float(np.abs(cam_gt - cam_edge_contrast).mean()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam.utils.image import show_cam_on_image

ov_edge_contrast = show_cam_on_image(img.astype(np.float32), cam_edge_contrast, use_rgb=True)
ov_gt = show_cam_on_image(img.astype(np.float32), cam_gt, use_rgb=True)

plt.figure(figsize=(14,7))
plt.subplot(2,3,1); plt.imshow(img); plt.axis("off"); plt.title("Input")
plt.subplot(2,3,2); plt.imshow(gt, cmap="gray"); plt.axis("off"); plt.title("GT mask")
plt.subplot(2,3,3); plt.imshow(ring, cmap="gray"); plt.axis("off"); plt.title("Boundary ring")

plt.subplot(2,3,4); plt.imshow(ov_gt); plt.axis("off"); plt.title("GradCAM (GT mean target)")
plt.subplot(2,3,5); plt.imshow(ov_edge_contrast); plt.axis("off"); plt.title("GradCAM (Boundary contrast target)")
plt.subplot(2,3,6); plt.imshow(np.abs(cam_gt - cam_edge_contrast), cmap="magma"); plt.axis("off"); plt.title("|GT - Contrast|")
plt.tight_layout()
plt.show()

In [ ]:
out_path = "/content/vgg_unet_boundary_contrast_cam.png"
plt.figure(figsize=(14,7))
plt.subplot(2,3,1); plt.imshow(img); plt.axis("off"); plt.title("Input")
plt.subplot(2,3,2); plt.imshow(gt, cmap="gray"); plt.axis("off"); plt.title("GT mask")
plt.subplot(2,3,3); plt.imshow(ring, cmap="gray"); plt.axis("off"); plt.title("Boundary ring")
plt.subplot(2,3,4); plt.imshow(ov_gt); plt.axis("off"); plt.title("GradCAM (GT mean target)")
plt.subplot(2,3,5); plt.imshow(ov_edge_contrast); plt.axis("off"); plt.title("GradCAM (Boundary contrast target)")
plt.subplot(2,3,6); plt.imshow(np.abs(cam_gt - cam_edge_contrast), cmap="magma"); plt.axis("off"); plt.title("|GT - Contrast|")
plt.tight_layout()
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", out_path)

In [ ]:


!pip -q install grad-cam opencv-python

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image


GRADCAM_DIR = os.path.join(SAVE_DIR, "gradcam_epochs_1_4_8_best")
os.makedirs(GRADCAM_DIR, exist_ok=True)

CHECKPOINTS = {
    "epoch_1": os.path.join(SAVE_DIR, "vgg_unet_epoch_1.pth"),
    "epoch_4": os.path.join(SAVE_DIR, "vgg_unet_epoch_4.pth"),
    "epoch_8": os.path.join(SAVE_DIR, "vgg_unet_epoch_8.pth"),
    "best": os.path.join(SAVE_DIR, "vgg_unet_best.pth")
}

CHECKPOINTS = {k: v for k, v in CHECKPOINTS.items() if os.path.exists(v)}

print("Available checkpoints:")
for name, path in CHECKPOINTS.items():
    print(name, "->", path)



denorm = transforms.Normalize(
    mean=(-0.485/0.229, -0.456/0.224, -0.406/0.225),
    std=(1/0.229, 1/0.224, 1/0.225)
)


def load_vgg_unet_checkpoint(path):
    temp_model = VGGUNet(pretrained=False).to(device)
    replace_relu_inplace(temp_model)

    temp_model.load_state_dict(torch.load(path, map_location=device), strict=False)
    temp_model.eval()
    return temp_model


class SegmentationMaskTargetMean:
    """
    Target για segmentation Grad-CAM.
    Αντί να εξηγούμε μια κλάση, εξηγούμε τη μέση ενεργοποίηση
    μέσα σε μια συγκεκριμένη περιοχή μάσκας.
    """
    def __init__(self, mask_2d):
        self.mask = torch.from_numpy(mask_2d.astype(np.float32))
        self.den = float(mask_2d.sum() + 1e-6)

    def __call__(self, model_output):
        logits = model_output[0, 0]
        mask = self.mask.to(logits.device)
        return (logits * mask).sum() / self.den


def make_boundary_ring(gt_mask, kernel_size=9):
    """
    Φτιάχνει boundary ring γύρω από τη βλάβη.
    Είναι χρήσιμο γιατί στο segmentation θέλουμε να δούμε
    αν το μοντέλο κοιτάζει τα όρια της βλάβης.
    """
    gt_u8 = (gt_mask * 255).astype(np.uint8)
    k = np.ones((kernel_size, kernel_size), np.uint8)

    dil = cv2.dilate(gt_u8, k, iterations=1)
    ero = cv2.erode(gt_u8, k, iterations=1)

    ring = ((dil > 0) & (ero == 0)).astype(np.float32)

    return ring


def get_target_layer_for_segmentation(model):
    """
    Για U-Net segmentation συνήθως δουλεύει καλύτερα ένα decoder layer,
    επειδή είναι πιο κοντά στην τελική χωρική μάσκα.
    """
    return model.dec2.net[3]



def compute_gradcam_for_model(temp_model, x1, gt_mask, method="gradcam"):
    temp_model.eval()
    torch.set_grad_enabled(True)

    with torch.enable_grad():
        logits = temp_model(x1)
        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred_mask = (prob > 0.5).astype(np.float32)

    ring = make_boundary_ring(gt_mask, kernel_size=9)

    target_layer = get_target_layer_for_segmentation(temp_model)

    if method == "gradcam++":
        cam_engine = GradCAMPlusPlus(model=temp_model, target_layers=[target_layer])
    else:
        cam_engine = GradCAM(model=temp_model, target_layers=[target_layer])

    cam_gt = cam_engine(
        input_tensor=x1,
        targets=[SegmentationMaskTargetMean(gt_mask)]
    )[0]

    cam_pred = cam_engine(
        input_tensor=x1,
        targets=[SegmentationMaskTargetMean(pred_mask)]
    )[0]

    cam_edge = cam_engine(
        input_tensor=x1,
        targets=[SegmentationMaskTargetMean(ring)]
    )[0]

    return prob, pred_mask, ring, cam_gt, cam_pred, cam_edge



xb, yb = next(iter(val_loader))
xb = xb.to(device)
yb = yb.to(device)

SAMPLE_INDEX = 0
x1 = xb[SAMPLE_INDEX:SAMPLE_INDEX+1]
gt = yb[SAMPLE_INDEX, 0].detach().cpu().numpy()

img = denorm(x1[0].detach().cpu()).permute(1, 2, 0).clamp(0, 1).numpy()



CAM_METHOD = "gradcam++"

n_rows = len(CHECKPOINTS)
plt.figure(figsize=(18, 4 * n_rows))

row = 0

for ckpt_name, ckpt_path in CHECKPOINTS.items():

    temp_model = load_vgg_unet_checkpoint(ckpt_path)

    prob, pred_mask, ring, cam_gt, cam_pred, cam_edge = compute_gradcam_for_model(
        temp_model=temp_model,
        x1=x1,
        gt_mask=gt,
        method=CAM_METHOD
    )

    ov_gt = show_cam_on_image(img.astype(np.float32), cam_gt, use_rgb=True)
    ov_pred = show_cam_on_image(img.astype(np.float32), cam_pred, use_rgb=True)
    ov_edge = show_cam_on_image(img.astype(np.float32), cam_edge, use_rgb=True)

    base = row * 6

    plt.subplot(n_rows, 6, base + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{ckpt_name}\nInput")

    plt.subplot(n_rows, 6, base + 2)
    plt.imshow(gt, cmap="gray")
    plt.axis("off")
    plt.title("GT mask")

    plt.subplot(n_rows, 6, base + 3)
    plt.imshow(prob, cmap="gray")
    plt.axis("off")
    plt.title("Pred prob")

    plt.subplot(n_rows, 6, base + 4)
    plt.imshow(ov_gt)
    plt.axis("off")
    plt.title("CAM: GT target")

    plt.subplot(n_rows, 6, base + 5)
    plt.imshow(ov_pred)
    plt.axis("off")
    plt.title("CAM: Pred target")

    plt.subplot(n_rows, 6, base + 6)
    plt.imshow(ov_edge)
    plt.axis("off")
    plt.title("CAM: Boundary target")

    del temp_model
    torch.cuda.empty_cache()

    row += 1

plt.tight_layout()

save_path = os.path.join(GRADCAM_DIR, f"vgg_unet_{CAM_METHOD}_epochs_1_4_8_best.png")
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", save_path)


In [ ]:
NUM_IMAGES = 4
CAM_METHOD = "gradcam++"

for sample_idx in range(NUM_IMAGES):

    xb, yb = next(iter(val_loader))
    xb = xb.to(device)
    yb = yb.to(device)

    x1 = xb[sample_idx:sample_idx+1]
    gt = yb[sample_idx, 0].detach().cpu().numpy()
    img = denorm(x1[0].detach().cpu()).permute(1, 2, 0).clamp(0, 1).numpy()

    n_rows = len(CHECKPOINTS)
    plt.figure(figsize=(18, 4 * n_rows))

    row = 0

    for ckpt_name, ckpt_path in CHECKPOINTS.items():

        temp_model = load_vgg_unet_checkpoint(ckpt_path)

        prob, pred_mask, ring, cam_gt, cam_pred, cam_edge = compute_gradcam_for_model(
            temp_model=temp_model,
            x1=x1,
            gt_mask=gt,
            method=CAM_METHOD
        )

        ov_gt = show_cam_on_image(img.astype(np.float32), cam_gt, use_rgb=True)
        ov_pred = show_cam_on_image(img.astype(np.float32), cam_pred, use_rgb=True)
        ov_edge = show_cam_on_image(img.astype(np.float32), cam_edge, use_rgb=True)

        base = row * 6

        plt.subplot(n_rows, 6, base + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{ckpt_name}\nInput")

        plt.subplot(n_rows, 6, base + 2)
        plt.imshow(gt, cmap="gray")
        plt.axis("off")
        plt.title("GT mask")

        plt.subplot(n_rows, 6, base + 3)
        plt.imshow(prob, cmap="gray")
        plt.axis("off")
        plt.title("Pred prob")

        plt.subplot(n_rows, 6, base + 4)
        plt.imshow(ov_gt)
        plt.axis("off")
        plt.title("CAM: GT target")

        plt.subplot(n_rows, 6, base + 5)
        plt.imshow(ov_pred)
        plt.axis("off")
        plt.title("CAM: Pred target")

        plt.subplot(n_rows, 6, base + 6)
        plt.imshow(ov_edge)
        plt.axis("off")
        plt.title("CAM: Boundary target")

        del temp_model
        torch.cuda.empty_cache()

        row += 1

    plt.tight_layout()

    save_path = os.path.join(
        GRADCAM_DIR,
        f"vgg_unet_{CAM_METHOD}_sample_{sample_idx}_epochs_1_4_8_best.png"
    )

    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

In [ ]:
import shap
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os

SHAP_DIR = os.path.join(SAVE_DIR, "shap_vgg_unet")
os.makedirs(SHAP_DIR, exist_ok=True)


class VGGUNetScalarWrapper(nn.Module):
    """
    Μετατρέπει την έξοδο του U-Net από μάσκα (B,1,H,W)
    σε scalar score (B,1), ώστε να μπορεί να δουλέψει το SHAP.

    Εδώ εξηγούμε τη μέση πιθανότητα της predicted lesion area.
    """
    def __init__(self, base_model, threshold=0.5):
        super().__init__()
        self.base_model = base_model
        self.threshold = threshold

    def forward(self, x):
        logits = self.base_model(x)

        score = logits.mean(dim=(1, 2, 3), keepdim=False)

        return score.unsqueeze(1)



shap_model = VGGUNet(pretrained=False).to(device)
replace_relu_inplace(shap_model)
shap_model.load_state_dict(torch.load(best_path, map_location=device), strict=False)
shap_model.eval()

scalar_model = VGGUNetScalarWrapper(shap_model).to(device)
scalar_model.eval()

torch.set_grad_enabled(True)



BACKGROUND_SIZE = 2
EXPLAIN_SIZE = 5

background_list = []
for i in range(BACKGROUND_SIZE):
    x_bg, _ = train_ds[i]
    background_list.append(x_bg)

background_tensor = torch.stack(background_list).to(device)


explain_list = []
explain_masks = []

for i in range(EXPLAIN_SIZE):
    x_ex, y_ex = val_ds[i]
    explain_list.append(x_ex)
    explain_masks.append(y_ex)

explain_tensor = torch.stack(explain_list).to(device)

print("background_tensor:", background_tensor.shape)
print("explain_tensor:", explain_tensor.shape)



explainer = shap.DeepExplainer(scalar_model, background_tensor)

shap_values = explainer.shap_values(explain_tensor)


if isinstance(shap_values, list):
    shap_values = shap_values[0]

print("SHAP values shape:", np.array(shap_values).shape)


In [ ]:


def tensor_to_rgb_image(x):
    img = denorm(x.detach().cpu()).permute(1, 2, 0).clamp(0, 1).numpy()
    return img



shap_np = np.array(shap_values)


if shap_np.ndim == 5 and shap_np.shape[-1] == 1:
    shap_np = shap_np.squeeze(axis=-1) # Now shape is (B, C, H, W)


if shap_np.shape[1] == 3:
    # Από (B,C,H,W) σε (B,H,W,C)
    shap_img = np.transpose(shap_np, (0, 2, 3, 1))
else:

    shap_img = shap_np

images_np = np.stack(
    [
        tensor_to_rgb_image(explain_tensor[i])
        for i in range(explain_tensor.shape[0])
    ]
)


shap_heatmaps = np.mean(np.abs(shap_img), axis=-1)

for i in range(explain_tensor.shape[0]):

    img = images_np[i]
    shap_map = shap_heatmaps[i]

    shap_map = shap_map - shap_map.min()
    shap_map = shap_map / (shap_map.max() + 1e-8)

    with torch.no_grad():
        logits = shap_model(explain_tensor[i : i + 1])
        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

    gt = explain_masks[i][0].detach().cpu().numpy()

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Input")

    plt.subplot(1, 4, 2)
    plt.imshow(gt, cmap="gray")
    plt.axis("off")
    plt.title("GT mask")

    plt.subplot(1, 4, 3)
    plt.imshow(prob, cmap="gray")
    plt.axis("off")
    plt.title("Pred prob")

    plt.subplot(1, 4, 4)
    plt.imshow(img)
    plt.imshow(shap_map, cmap="magma", alpha=0.45)
    plt.axis("off")
    plt.title("SHAP overlay")

    plt.tight_layout()

    save_path = os.path.join(SHAP_DIR, f"shap_vgg_unet_sample_{i}.png")
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)
